# getting extents for PNG from LCCS BCEs
- load in PNG province files (prepared by Chloe)
- get LCCS for extent (through geobox)
- get seagrass extent (through geobox)
- could pixels of BCEs and put into table

In [1]:
import os, sys
import numpy as np
import pandas as pd
import geopandas as gpd

import fiona
from shapely.geometry import shape

import rioxarray
from rasterio.features import geometry_mask

import datacube
dc = datacube.Datacube(app="extents")
from datacube.utils.aws import configure_s3_access
from datacube.utils.geometry import Geometry
from dea_tools.datahandling import load_reproject

# 8.4.25 - Matt Paget work around for error CPLE_HttpResponseError: CURL error: Failed to connect to easi-caching-proxy.caching-proxy port 80 after 0 ms: Couldn't connect to server
sys.path.insert(1, "/home/jovyan/code/easi-notebooks/")
from easi_tools.notebook_utils import unset_cachingproxy


In [2]:
# Access AWS "requester-pays" buckets
# This is necessary for reading data from most third-party AWS S3 buckets such as for Landsat and Sentinel-2
configure_s3_access(aws_unsigned=False, requester_pays=True);

In [3]:
# This defines the function that converts a linear vector file into a string of x,y coordinates

def geom_query(geom, geom_crs='EPSG:32755'):
    """
    Create datacube query snippet for geometry
    """
    return {
        'x': (geom.bounds[0], geom.bounds[2]),
        'y': (geom.bounds[1], geom.bounds[3]),
        'crs': geom_crs
    }

def warp_geometry(geom, crs_crs, dst_crs):
    """
    warp geometry from crs_crs to dst_crs
    """
    return shapely.geometry.shape(rasterio.warp.transform_geom(crs_crs, dst_crs, shapely.geometry.mapping(geom)))

In [4]:
crs = "EPSG:32755"
res = (30, -30)

query =({'output_crs':crs,
         'resolution':res})

In [5]:
# Define the folder containing the .shp files
folder_path = '../data/PNG_province_TSZbuffered_EPSG32755/'

# List to hold individual GeoDataFrames
gdfs = []

# Loop through all .shp files in the folder
for shp_file in os.listdir(folder_path):
    if shp_file.endswith('.shp'):
        # Load the shapefile into a GeoDataFrame
        transects_selection = gpd.read_file(os.path.join(folder_path, shp_file))
        
        # Extract the Id from the filename
        file_id = shp_file.split('_TSZ')[0]
        
        # Keep only the geometry column and add the Id column
        transects_selection = transects_selection[['geometry']]
        transects_selection['Id'] = file_id
        
        print(f'Processing file: {shp_file}')
        
        # Append the GeoDataFrame to the list
        gdfs.append(transects_selection)

# Concatenate all GeoDataFrames into one
combined_gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))

Processing file: Central_prov_TSZ.shp
Processing file: New_Ireland_TSZ_buffer.shp
Processing file: Manus_TSZ_buffer.shp
Processing file: National_Captial_District_TSZ.shp
Processing file: East_Sepik_TSZ.shp
Processing file: West_New_Britain_TSZ_buffer_Erase.shp
Processing file: Gulf_prov_TSZ.shp
Processing file: Morobe_prov_TSZ.shp
Processing file: Bougainville_TSZ_buffer.shp
Processing file: Milne_Bay_TSZ_buff.shp
Processing file: West_Sepik_TSZ.shp
Processing file: Western_prov_TSZ.shp
Processing file: Madang_prov_TSZ.shp
Processing file: East_New_Britain_TSZ_buffer_erase.shp
Processing file: Oro_prov_TSZ.shp


In [6]:
# combined_gdf = combined_gdf.iloc[5:]
# combined_gdf

In [7]:
# Initialize an empty DataFrame to store the results
results_df = pd.DataFrame(columns=['Province', 'Value', 'BCE', 'Area_km2'])


# Loop through polygons in geodataframe and add geom to queries
for index, row in combined_gdf.iterrows():
    print(f'Feature: {index + 1}/{len(combined_gdf)}')

    # Extract the feature's geometry as a datacube geometry object
    geom = Geometry(geom=row.geometry, crs=combined_gdf.crs)

    # Update the query to include the geopolygon
    query.update({'geopolygon': geom})

    # load in GLO data 
    with unset_cachingproxy():
        GLO30 = dc.load(product='copernicus_dem_30', resampling='bilinear', time = ('2022-01-01', '2022-12-31'), **query)

    # load LCCS band 5 for BCEs
    # LCCS data EPSG:32755 30m pixel
    LCCS_path = '/home/jovyan/code/livingearth_png/notebooks/png_lccs_classification_v0_2_data_merged.tif'
    LCCS_load = load_reproject(path=LCCS_path, how=GLO30.odc.geobox).load()
    LCCS_load = LCCS_load.rename('lccs')

    # Clip the DataArray to the GeoDataFrame
    lccs_clipped = LCCS_load.rio.clip([row.geometry], combined_gdf.crs, drop=False, invert=False)

    # Mask out values outside polygon with NaNs (if not already)
    lccs_clipped = lccs_clipped.where(~lccs_clipped.isnull(), other=np.nan)
    
    # # load seagrass ### NEED TO COMPLETE
    # # LCCS data EPSG:32755 30m pixel
    # seagrass_path = '/home/jovyan/code/livingearth_png/manuscript_figures/mosaic_cog_Seagrass_B2_extent_z56.tif'
    # seagrass_load = load_reproject(path=seagrass_path, how=GLO30.odc.geobox).load()
    # seagrass_load = seagrass_load.rename('seagrass')
    
    # Assuming lccs is your DataArray
    unique_values, counts = np.unique(lccs_clipped.values, return_counts=True)

    # Filter for values 1, 2, and 3
    mask = np.isin(unique_values, [1, 2, 3])
    filtered_values = unique_values[mask]
    filtered_counts = counts[mask]
    
    # Multiply counts by 0.09 and 0.0009 to get area (in ha and km²)
    filtered_areas_ha = filtered_counts * 0.09
    filtered_areas_km = filtered_counts * 0.0009
    
    # Map values to categories
    value_to_category = {1: 'mangrove', 2: 'supratidal', 3: 'saltmarsh'}
    categories = [value_to_category[val] for val in filtered_values]

    # Create a DataFrame for the current feature
    feature_df = pd.DataFrame({
        'Province': row['Id'],
        'BCE': categories,
        'Count': filtered_counts,
        'Area_ha': filtered_areas_ha,
        'Area_km2': filtered_areas_km
    })
    
    # Append the feature DataFrame to the results DataFrame
    results_df = pd.concat([results_df, feature_df], ignore_index=True)

Feature: 1/15


/env/lib/python3.12/site-packages/rasterio/warp.py:344: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  _reproject(


Feature: 2/15
Feature: 3/15
Feature: 4/15
Feature: 5/15
Feature: 6/15
Feature: 7/15
Feature: 8/15
Feature: 9/15
Feature: 10/15
Feature: 11/15
Feature: 12/15
Feature: 13/15
Feature: 14/15
Feature: 15/15


In [8]:
# Ensure the 'Category' column is a categorical type with the desired order
results_df['BCE'] = pd.Categorical(results_df['BCE'], categories=['mangrove', 'saltmarsh', 'supratidal'], ordered=True)

# Sort the DataFrame by the 'Category' column
extents_df = results_df.sort_values('BCE').reset_index(drop=True)


In [9]:
extents_df

,Province,Value,BCE,Area_km2,Count,Area_ha
0,Central_prov,NaN,mangrove,494.2629,549181.0,49426.29
1,Bougainville,NaN,mangrove,39.3984,43776.0,3939.84
2,Gulf_prov,NaN,mangrove,2028.6468,2254052.0,202864.68
3,Milne_Bay,NaN,mangrove,314.4681,349409.0,31446.81
4,West_New_Britain,NaN,mangrove,120.4803,133867.0,12048.03
5,West_Sepik,NaN,mangrove,9.1197,10133.0,911.97
6,East_Sepik,NaN,mangrove,151.0209,167801.0,15102.09
7,National_Captial_District,NaN,mangrove,1.3410,1490.0,134.10
8,Western_prov,NaN,mangrove,707.4819,786091.0,70748.19
9,Madang_prov,NaN,mangrove,10.0935,11215.0,1009.35


In [10]:
# LCCS_load.odc.write_cog('LCCS_load_Gulf_EPSG32755.tif', overwrite=True)

In [11]:
#### SEAGRASS, not integrated to above at the moment ####


# Initialize an empty DataFrame to store the results
results_df = pd.DataFrame(columns=['Province', 'Value', 'BCE', 'Area_km2'])


# Loop through polygons in geodataframe and add geom to queries
for index, row in combined_gdf.iterrows():
    print(f'Feature: {index + 1}/{len(combined_gdf)}')

    # Extract the feature's geometry as a datacube geometry object
    geom = Geometry(geom=row.geometry, crs=combined_gdf.crs)

    # Update the query to include the geopolygon
    query.update({'geopolygon': geom})

    # load in GLO data 
    with unset_cachingproxy():
        GLO30 = dc.load(product='copernicus_dem_30', resampling='bilinear', time = ('2022-01-01', '2022-12-31'), **query)

    # load seagrass
    # LCCS data EPSG:32755 30m pixel
    seagrass_path = '/home/jovyan/code/livingearth_png/manuscript_figures/data/mosaic_cog_Seagrass_B2_extent_z56.tif'
    seagrass_load = load_reproject(path=seagrass_path, how=GLO30.odc.geobox).load()
    seagrass_load = seagrass_load.rename('seagrass')

    # Clip the DataArray to the GeoDataFrame
    seagrass_clipped = seagrass_load.rio.clip([row.geometry], combined_gdf.crs, drop=False, invert=False)

    # Mask out values outside polygon with NaNs (if not already)
    seagrass_clipped = seagrass_clipped.where(~seagrass_clipped.isnull(), other=np.nan)
    

    
    # Assuming lccs is your DataArray
    unique_values, counts = np.unique(seagrass_clipped.values, return_counts=True)

    # Filter for values 1, 2, and 3
    mask = np.isin(unique_values, [1])
    filtered_values = unique_values[mask]
    filtered_counts = counts[mask]
    
    # Multiply counts by 0.09 and 0.0009 to get area (in ha and km²)
    filtered_areas_ha = filtered_counts * 0.09
    filtered_areas_km = filtered_counts * 0.0009
    
    # Map values to categories
    value_to_category = {1: 'seagrass'}
    categories = [value_to_category[val] for val in filtered_values]

    # Create a DataFrame for the current feature
    feature_df = pd.DataFrame({
        'Province': row['Id'],
        'BCE': categories,
        'Count': filtered_counts,
        'Area_ha': filtered_areas_ha,
        'Area_km2': filtered_areas_km
    })
    
    # Append the feature DataFrame to the results DataFrame
    results_df = pd.concat([results_df, feature_df], ignore_index=True)

Feature: 1/15
Feature: 2/15
Feature: 3/15
Feature: 4/15
Feature: 5/15
Feature: 6/15
Feature: 7/15
Feature: 8/15
Feature: 9/15
Feature: 10/15
Feature: 11/15
Feature: 12/15
Feature: 13/15
Feature: 14/15
Feature: 15/15


In [12]:
results_df

,Province,Value,BCE,Area_km2,Count,Area_ha
0,Central_prov,NaN,seagrass,8.6742,9638.0,867.42
1,New_Ireland,NaN,seagrass,57.0825,63425.0,5708.25
2,Manus,NaN,seagrass,34.7625,38625.0,3476.25
3,National_Captial_District,NaN,seagrass,2.6766,2974.0,267.66
4,East_Sepik,NaN,seagrass,1.5057,1673.0,150.57
5,West_New_Britain,NaN,seagrass,73.0494,81166.0,7304.94
6,Morobe_prov,NaN,seagrass,7.7445,8605.0,774.45
7,Bougainville,NaN,seagrass,17.8398,19822.0,1783.98
8,Milne_Bay,NaN,seagrass,83.5821,92869.0,8358.21
9,West_Sepik,NaN,seagrass,0.1224,136.0,12.24
